In [ ]:
!pip install pypdf transformers bitsandbytes accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
import os

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Successfully uploaded: {file_name}")

Saving ft.docx to ft.docx
Successfully uploaded: ft.docx


In [ ]:
!pip install pypdf python-docx -q
!apt-get install -y catdoc -q # Linux utility to read older .doc files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.3 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  catdoc
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 89.5 kB of archives.
After this operation, 695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 catdoc amd64 1:0.95-5 [89.5 kB]
Fetched 89.5 kB in 0s (1,912 kB/s)
Selecting previously unselected package catdoc.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../catdoc_1%3a0.95-5_amd64.deb ...
Unpacking catdoc (1:0.95-5) ...
Setting up catdoc (1:0.95-5) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import os
import subprocess
from pypdf import PdfReader
from docx import Document

def extract_text(file_path):
    # 1. Handle PDF files
    if file_path.endswith('.pdf'):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text

    # 2. Handle modern Word files (.docx)
    elif file_path.endswith('.docx'):
        doc = Document(file_path)
        text = []
        for paragraph in doc.paragraphs:
            text.append(paragraph.text)
        return "\n".join(text)

    # 3. Handle legacy Word files (.doc)
    elif file_path.endswith('.doc'):
        # Uses the Linux 'catdoc' tool we installed to extract plain text from old binary .doc
        result = subprocess.run(['catdoc', file_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            return result.stdout
        else:
            raise ValueError(f"Could not read legacy .doc file. Error: {result.stderr}")

    # 4. Fallback for plain text or configuration files (.txt, .md, etc.)
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()

# Load your file content (works with .pdf, .docx, .doc, .txt)
document_context = extract_text(file_name)
print(f"Extracted {len(document_context)} characters from the document.")

Extracted 1393 characters from the document.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Configure 4-bit quantization to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Load the model onto the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def ask_about_document(user_question):
    # Constructing a structured system message to restrict the model's knowledge to your file
    messages = [
        {
            "role": "system",
            "content": f"You are a helpful assistant. Answer the user's question accurately using ONLY the following reference document:\n\n{document_context}"
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    # Format the prompt into tokens using Qwen's specific template structure
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    # Generate the text response
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=500,  # Limits response length (increase if you want longer answers)
        temperature=0.1,     # Keeps the model focused and factual based on your text
        do_sample=False
    )

    # Clean up the output so it only displays the answer (skipping the original prompt)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
import sys

def chat_with_document():
    print("✨ Document Chat Activated! Type your question below.")
    print("👉 Type 'exit' or 'quit' to stop.\n")
    print("-" * 50)

    while True:
        # 1. Take continuous user input
        user_question = input("👤 You: ")

        if user_question.lower() in ['exit', 'quit']:
            print("👋 Chat ended. Happy building!")
            break

        if not user_question.strip():
            continue

        print("\n🤖 Thinking...")

        # 2. Format the payload for Qwen
        messages = [
            {
                "role": "system",
                "content": f"You are a strict, factual assistant. Answer the user's question accurately using ONLY the information provided in this text:\n\n{document_context}"
            },
            {
                "role": "user",
                "content": user_question
            }
        ]

        # 3. Tokenize and generate response on the GPU
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=500,  # Room for detailed answers
            temperature=0.1,     # Keeps it tightly anchored to your text file
            do_sample=False
        )

        # 4. Decode and print output
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        print(f"\n🤖 Qwen:\n{response}")
        print("-" * 50)

# Start the interactive loop!
chat_with_document()

✨ Document Chat Activated! Type your question below.
👉 Type 'exit' or 'quit' to stop.

--------------------------------------------------
👤 You: what if forest


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🤖 Thinking...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



🤖 Qwen:
forest
--------------------------------------------------
👤 You: who won the race

🤖 Thinking...

🤖 Qwen:
The hare won the race.
--------------------------------------------------
👤 You: what is hare

🤖 Thinking...

🤖 Qwen:
A hare is a small mammal belonging to the family Leporidae. They are known for their ability to run at high speeds, typically around 10-25 miles per hour (16-40 km/h). Hares are found throughout much of the world, but they are most commonly found in North America, Europe, Asia, and parts of Africa.
--------------------------------------------------
👤 You: who took quick nap

🤖 Thinking...

🤖 Qwen:
The hare took a quick nap in the cool shade of a large tree.
--------------------------------------------------
👤 You: what does cool shade mean

🤖 Thinking...

🤖 Qwen:
A cool shade is a place where there is a low temperature or humidity level, usually found near trees or other plants. It provides a comfortable environment for people to sit and relax while enjoyin